[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/11_sft_grpo_unsloth.ipynb)

# 11 · SFT + GRPO on Qwen3（照 Unsloth 官方 notebook 的写法）

结构参考 Unsloth 的两个官方 Colab：
[Qwen3 (4B) Instruct（SFT）](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-Instruct.ipynb) 和
[Qwen3 (4B) GRPO](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-GRPO.ipynb)：
同一个模型对象开 `fast_inference=True`（内置 vLLM），先做一小段 **SFT 预热**（学会格式），再做 **GRPO**（可验证奖励）。

我们在这个流程上加了**同一个 vLLM 引擎里的 GPQA 快速评测**，在三个时间点各测一次：
`0_base`（不加 LoRA）→ `1_sft`（SFT LoRA）→ `2_grpo`（GRPO LoRA），直接看到每一步的变化。

| 模式 | 数据量 | 评测 | 预计用时（T4，0.6B，待实测） |
|---|---|---|---|
| `quick` | SFT 200 条 Dolci + 200 条选择题格式样本，40 步；GRPO 30 步 | GPQA Diamond 前 60 题，最长 1024 token | 约 15–25 分钟 |
| `medium` | SFT 1000 + 500 条，150 步；GRPO 150 步 | GPQA Diamond 全部 198 题，最长 2048 token | 1.7B 约 1 小时 |
| `full` | SFT 3000 + 1000 条，1 epoch；GRPO 300 步 | GPQA Diamond 全部 198 题，最长 2048 token | 约 1.5–3 小时 |

⚠️ 这里的快速评测**不是**排行榜分数：题目子集更小、生成更短、选项顺序与 lm-eval 不同。它只用来比较同一次运行里 base / SFT / GRPO 的相对变化。
正式分数请把合并后的模型推到 Hub，再在新的 runtime 里用 `main` 分支的 `01_eval_gpqa.ipynb` 评测。

**模型大小怎么选**（Qwen 官方后训练版的 GPQA Diamond，随机猜是 25%）：

| 模型 | 官方 non-thinking | 官方 thinking | 用途 |
|---|---|---|---|
| Qwen3-0.6B | 22.9 | 27.9 | 只适合**验证流程**：官方完整后训练都只在随机水平附近，能看到的主要是格式变化 |
| Qwen3-1.7B | 28.6 | 40.1 | **能看到变化的最小尺寸**，提升主要来自长推理 |
| Qwen3-4B | 41.7 | 55.9 | 变化最明显；Unsloth 的官方 T4 GRPO 示例就是这个尺寸 |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆提交分支（数据准备 / 去污染脚本在里面）
import os
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
%cd GPQA-eval
!git pull -q

In [ ]:
# 安装（约 3–5 分钟）。输出不隐藏：如果安装报错请看这里。
# 与 Unsloth 官方 GRPO notebook 的区别：不再在 T4 上强装 vllm==0.11.2（当前 Colab 的 Python 3.13 / torch 2.14 上装不上），
# 而是让 uv 一起解析 unsloth + vllm 的兼容版本（2026-11 实测：unsloth 2026.11.3 + vllm 0.33.0 + trl 1.15 + transformers 5.17 在 T4 上跑通）。
!pip install --upgrade -q uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install unsloth vllm {_numpy} {_pil}

In [ ]:
import os, importlib.metadata as md, torch
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"   # Unsloth：GRPO 时让 vLLM 和训练共享显存
# T4（计算能力 7.5）不支持 FlashInfer 采样器。Unsloth 会给 vLLM 打开它，新版 vLLM 随即报
# "FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5"。这里关掉。
if torch.cuda.get_device_capability()[0] < 8:
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    print("T4 detected: FlashInfer disabled for vLLM")
VERSIONS = {}
for p in ["unsloth", "vllm", "trl", "transformers", "torch", "peft"]:
    try: VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError: VERSIONS[p] = None
print(VERSIONS)

In [ ]:
# GPQA 是 gated 数据集：先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款；Colab 左侧 🔑 里放 HF_TOKEN
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

## 0. 配置

In [ ]:
MODE         = "quick"                        # "quick"（验证流程）| "medium"（1.7B 看变化）| "full"
MODEL_NAME   = "unsloth/Qwen3-0.6B-Base"      # 看到真实变化：换成 "unsloth/Qwen3-1.7B-Base" 或 "unsloth/Qwen3-4B-Base"
BASE_MODEL   = "Qwen/Qwen3-0.6B-Base"         # 提交时填的原始 base（与 MODEL_NAME 对应）
LOAD_IN_4BIT = False                          # 16-bit LoRA（Unsloth 官方 GRPO 示例也是 False）。
                                              # 想用 bnb-4bit：MODEL_NAME="unsloth/Qwen3-0.6B-Base-unsloth-bnb-4bit"，这里改 True
MAX_SEQ_LEN  = 2048
LORA_RANK    = 32
GPU_UTIL     = 0.7                            # vLLM 占用比例；OOM 就调小（4B 可参考 Unsloth 的 0.9 + standby）

DOLCI_DATASET = "allenai/Dolci-Instruct-SFT-7B"   # 也可以试 "allenai/Dolci-Think-SFT-7B"（思维链更长，会被长度过滤掉很多）
DOLCI_INCLUDE = []                                # 例如 ["math", "science"]；空 = 不按来源过滤（最快）
DOLCI_EXCLUDE = ["safety", "wildguard"]

Q = dict(
    quick=dict(sft_dolci_n=200, sft_mcq_n=200, sft_steps=40, grpo_prompts=200, grpo_steps=30,
               grpo_max_completion=512, eval_n=60, eval_max_tokens=1024, shuffle_buffer=1000),
    medium=dict(sft_dolci_n=1000, sft_mcq_n=500, sft_steps=150, grpo_prompts=1200, grpo_steps=150,
                grpo_max_completion=768, eval_n=None, eval_max_tokens=2048, shuffle_buffer=5000),
    full=dict(sft_dolci_n=3000, sft_mcq_n=1000, sft_steps=-1, grpo_prompts=3000, grpo_steps=300,
              grpo_max_completion=1024, eval_n=None, eval_max_tokens=2048, shuffle_buffer=20000),
)[MODE]
print(MODE, Q)

## 1. 加载模型（`fast_inference=True` = 训练和生成共用一个 vLLM 引擎）

In [ ]:
import os, torch
if torch.cuda.get_device_capability()[0] < 8:   # 再设一次，防止跳过了前面的格（T4 不支持 FlashInfer 采样器）
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = LOAD_IN_4BIT,
    fast_inference = True,          # 内置 vLLM，用于 GRPO 采样和下面的快速评测
    max_lora_rank = LORA_RANK,
    gpu_memory_utilization = GPU_UTIL,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_RANK * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
# 用 Qwen3 自带的 ChatML 模板（锁定评测 lm-eval 也会用保存下来的这个模板），不像 Unsloth GRPO 示例那样换成自定义标签
if tokenizer.chat_template is None:     # unsloth/Qwen3-*-Base 的 tokenizer 没有模板（run1 实测）
    from unsloth.chat_templates import get_chat_template
    tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-instruct")

# 关键：Base tokenizer 的 eos 是 <|endoftext|>，但 ChatML 每轮以 <|im_end|> 结束。
# 不改的话模型答完不会停（run1：SFT 后 100% 打满 token 上限，GRPO 每个回答都是 1024 token、格式奖励≈0）。
# TRL 用 tokenizer.eos_token_id + generation_config.eos_token_id 判断结束；保存的模型也带上这个设置，lm-eval 评测时才会停。
IM_END = tokenizer.convert_tokens_to_ids("<|im_end|>")
ENDOFTEXT = tokenizer.convert_tokens_to_ids("<|endoftext|>")
STOP_IDS = [IM_END, ENDOFTEXT]
tokenizer.eos_token = "<|im_end|>"
model.generation_config.eos_token_id = STOP_IDS
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    tokenizer.pad_token = "<|endoftext|>"
_demo = tokenizer.apply_chat_template([{"role": "user", "content": "Hi"}, {"role": "assistant", "content": "Hello"}], tokenize = False)
print(repr(_demo))
assert _demo.rstrip().endswith("<|im_end|>"), "chat template 应以 <|im_end|> 结束每轮"
print("eos:", tokenizer.eos_token, "stop ids:", STOP_IDS, "pad:", tokenizer.pad_token)

## 2. GPQA 快速评测（同一个 vLLM 引擎，只输出聚合数字）

题面与 lm-eval 的 `gpqa_diamond_cot_zeroshot` 模板一致；答案提取用两种方式：
`strict` = 出现 `The answer is (X)`，`flex` = 输出中最后一个 `(A)`–`(D)`（排行榜用的就是 flex）。

In [ ]:
import random, re, json, statistics, time
from collections import Counter
from datasets import load_dataset
from vllm import SamplingParams

TEMPLATE = ("What is the correct answer to this question:{q}\nChoices:\n"
            "(A) {a}\n(B) {b}\n(C) {c}\n(D) {d}\nLet's think step by step: ")
STRICT_RE = re.compile(r"The answer is \(([A-D])\)")
LETTER_RE = re.compile(r"\(([A-D])\)")

def _clean(t):
    return " ".join(str(t).replace(" [title]", ". ").split())

def build_gpqa(config, n=None):
    ds = load_dataset("Idavidrein/gpqa", config, split = "train")
    rows = []
    for i, d in enumerate(ds):
        choices = [_clean(d["Incorrect Answer 1"]), _clean(d["Incorrect Answer 2"]),
                   _clean(d["Incorrect Answer 3"]), _clean(d["Correct Answer"])]
        correct = choices[3]
        random.Random(1000 + i).shuffle(choices)
        rows.append((TEMPLATE.format(q = _clean(d["Question"]), a = choices[0], b = choices[1],
                                     c = choices[2], d = choices[3]), "ABCD"[choices.index(correct)]))
    return rows[:n] if n else rows

GPQA = build_gpqa("gpqa_diamond", Q["eval_n"])
GPQA_TEXTS = [tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize = False,
                                            add_generation_prompt = True) for p, _ in GPQA]
RESULTS = {}

def quick_eval(tag, lora_request = None):
    sp = SamplingParams(temperature = 0.6, top_p = 0.95, max_tokens = Q["eval_max_tokens"], seed = 0,
                        stop_token_ids = STOP_IDS)
    t0 = time.time()
    outs = model.fast_generate(GPQA_TEXTS, sampling_params = sp, lora_request = lora_request)
    strict = flex = extracted = truncated = 0
    lengths, letters = [], Counter()
    for o, (_, gold) in zip(outs, GPQA):
        c = o.outputs[0]
        m = STRICT_RE.findall(c.text)
        f = LETTER_RE.findall(c.text)
        strict += bool(m) and m[-1] == gold
        flex += bool(f) and f[-1] == gold
        extracted += bool(f)
        truncated += c.finish_reason == "length"
        lengths.append(len(c.token_ids))
        letters[f[-1] if f else "none"] += 1
    n = len(GPQA)
    RESULTS[tag] = dict(
        n = n, acc_flex = round(100 * flex / n, 1), acc_strict = round(100 * strict / n, 1),
        extract_rate = round(100 * extracted / n, 1), truncation_rate = round(100 * truncated / n, 1),
        mean_tokens = round(statistics.fmean(lengths)), letters = dict(sorted(letters.items())),
        minutes = round((time.time() - t0) / 60, 1),
    )
    print(tag, RESULTS[tag])

quick_eval("0_base")

## 3. SFT 预热（照 Unsloth Instruct notebook：chat 模板转成 `text` + `train_on_responses_only`）

两部分数据：
1. **Dolci**（OLMo 3 的 SFT 数据）：流式读取、按长度过滤到 `MAX_SEQ_LEN/2` 以内（和 Unsloth GRPO 示例一样）。
2. **选择题格式样本**：SciQ 训练集，题面套用 GPQA 模板，答案是 SciQ 的 `support` 段落 + `The answer is (X).`。
   这部分教会模型评测要的作答格式。SciQ 与 GPQA 无关，但仍然走一遍去污染检查。

In [ ]:
import os
os.makedirs("data", exist_ok = True)
inc = " ".join(DOLCI_INCLUDE); exc = " ".join(DOLCI_EXCLUDE)
inc_arg = f"--include {inc}" if inc else ""
N_DOLCI, HALF_LEN, SHUF = Q["sft_dolci_n"], MAX_SEQ_LEN // 2, Q["shuffle_buffer"]
!python scripts/prepare_dolci_sft.py --dataset {DOLCI_DATASET} --out data/dolci.jsonl --n {N_DOLCI} --tokenizer {MODEL_NAME} --max-tokens {HALF_LEN} --shuffle-buffer {SHUF} {inc_arg} --exclude {exc}

In [ ]:
SCIQ = load_dataset("allenai/sciq", split = "train").shuffle(seed = 0)

def sciq_mcq(ex, idx):
    rng = random.Random(idx)
    choices = [ex["distractor1"], ex["distractor2"], ex["distractor3"], ex["correct_answer"]]
    rng.shuffle(choices)
    letter = "ABCD"[choices.index(ex["correct_answer"])]
    prompt = TEMPLATE.format(q = ex["question"], a = choices[0], b = choices[1], c = choices[2], d = choices[3])
    return prompt, letter

# SciQ 前一段给 SFT，后一段给 GRPO，互不重叠
sft_mcq = []
for i, ex in enumerate(SCIQ):
    if len(sft_mcq) >= Q["sft_mcq_n"]:
        break
    if ex["support"].strip():
        prompt, letter = sciq_mcq(ex, i)
        sft_mcq.append({"messages": [{"role": "user", "content": prompt},
                                     {"role": "assistant", "content": ex["support"].strip() + f"\n\nThe answer is ({letter})."}]})
SCIQ_GRPO_START = i

dolci = [json.loads(l) for l in open("data/dolci.jsonl")]
sft_rows = [{"messages": r["prompt"] + r["completion"]} for r in dolci] + sft_mcq
with open("data/sft_all.jsonl", "w") as f:
    for r in sft_rows:
        f.write(json.dumps(r, ensure_ascii = False) + "\n")
!python scripts/decontam.py --train-jsonl data/sft_all.jsonl --out data/decontam_sft.json > /dev/null
bad = set(json.load(open("data/decontam_sft.json"))["flagged_row_indices"])
sft_rows = [r for i, r in enumerate(sft_rows) if i not in bad]
random.Random(0).shuffle(sft_rows)
DATA = dict(dolci = len(dolci), sciq_format = len(sft_mcq), dropped_decontam = len(bad))
print(DATA)

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# 以 <|im_end|>（现在的 eos）结尾，避免 TRL 再额外追加一个 eos
sft_ds = Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize = False).rstrip("\n")}
                            for r in sft_rows])
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_steps = 5,
        max_steps = Q["sft_steps"],
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = "outputs/sft",
    ),
)
# 只在 assistant 回复上算 loss（Qwen3 ChatML 标记）
trainer = train_on_responses_only(trainer, instruction_part = "<|im_start|>user\n",
                                  response_part = "<|im_start|>assistant\n")
sft_stats = trainer.train()
sft_losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
SFT = dict(steps = trainer.state.global_step, minutes = round(sft_stats.metrics["train_runtime"] / 60, 1),
           loss_first = sft_losses[0] if sft_losses else None, loss_last = sft_losses[-1] if sft_losses else None)
print(SFT)

model.save_lora("sft_lora")
quick_eval("1_sft", model.load_lora("sft_lora"))

## 4. GRPO（照 Unsloth GRPO notebook：`vllm_sampling_params` + 多个奖励函数）

题目：SciQ 中没用于 SFT 的部分，题面同样套 GPQA 模板。奖励：
- `format_reward`：出现且只出现一次 `The answer is (X)` → +1.0
- `correctness_reward`：最后一个 `(A)`–`(D)` 等于标准答案 → +2.0；输出里完全没有选项字母 → −1.0

In [ ]:
import numpy as np
del sft_ds, trainer
torch.cuda.empty_cache(); import gc; gc.collect()

grpo_rows = []
for j in range(SCIQ_GRPO_START, len(SCIQ)):
    if len(grpo_rows) >= Q["grpo_prompts"]:
        break
    prompt, letter = sciq_mcq(SCIQ[j], j)
    grpo_rows.append({"prompt": [{"role": "user", "content": prompt}], "answer": letter})
grpo_ds = Dataset.from_list(grpo_rows)

# 当前 TRL 不再支持 max_prompt_length（Unsloth 会提示忽略），所以像 Unsloth 示例那样直接过滤掉最长的 10% 题目
def _n_tokens(msgs):
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt = True, tokenize = True)
    return len(ids["input_ids"] if hasattr(ids, "keys") else ids)
lens = [_n_tokens(r["prompt"]) for r in grpo_rows]
max_prompt_length = int(np.quantile(lens, 0.9)) + 1
grpo_ds = grpo_ds.select(np.where(np.array(lens) < max_prompt_length)[0])
max_completion_length = min(Q["grpo_max_completion"], MAX_SEQ_LEN - max_prompt_length)
print(len(grpo_ds), "prompts; max_prompt_length", max_prompt_length, "max_completion_length", max_completion_length)

def format_reward(completions, **kwargs):
    return [1.0 if len(STRICT_RE.findall(c[0]["content"])) == 1 else 0.0 for c in completions]

def correctness_reward(completions, answer, **kwargs):
    out = []
    for c, a in zip(completions, answer):
        found = LETTER_RE.findall(c[0]["content"])
        out.append(-1.0 if not found else (2.0 if found[-1] == a else 0.0))
    return out

In [ ]:
from trl import GRPOConfig, GRPOTrainer

vllm_sampling_params = SamplingParams(
    min_p = 0.1, top_p = 1.0, top_k = -1, seed = 3407,
    stop_token_ids = STOP_IDS,          # 在 <|im_end|> 处停（run1 没停住，回答全是 1024 token）
)
NUM_GEN = 8
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    generation_kwargs = {"stop_token_ids": STOP_IDS},
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.001,
    warmup_steps = max(1, Q["grpo_steps"] // 10),
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = NUM_GEN,     # 每步 = 2 道题 × 8 个采样
    gradient_accumulation_steps = 2,
    num_generations = NUM_GEN,
    max_completion_length = max_completion_length,
    # loss_type / beta 不设：Unsloth 当前默认 loss_type="dapo", beta=0.0（无 KL）
    max_steps = Q["grpo_steps"],
    save_steps = 10_000,
    report_to = "none",
    output_dir = "outputs/grpo",
)
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [format_reward, correctness_reward],
    args = training_args,
    train_dataset = grpo_ds,
)
grpo_stats = trainer.train()

def _avg(key, part):
    vals = [h[key] for h in trainer.state.log_history if key in h]
    k = max(1, len(vals) // 5)
    return round(float(np.mean(vals[:k] if part == "first" else vals[-k:])), 3) if vals else None

GRPO = dict(steps = trainer.state.global_step, minutes = round(grpo_stats.metrics["train_runtime"] / 60, 1))
for key in ["reward", "rewards/format_reward/mean", "rewards/correctness_reward/mean", "completion_length",
            "completions/mean_length"]:
    if any(key in h for h in trainer.state.log_history):
        GRPO[key] = {"first": _avg(key, "first"), "last": _avg(key, "last")}
print(GRPO)

model.save_lora("grpo_lora")
quick_eval("2_grpo", model.load_lora("grpo_lora"))

## 5. 汇总（把最后打印的 JSON 整段贴回给 Claude）

In [ ]:
print(f"{'stage':8s} {'acc_flex':>8s} {'acc_strict':>10s} {'extract%':>9s} {'trunc%':>7s} {'tokens':>7s} {'eval_min':>8s}")
for k, v in RESULTS.items():
    print(f"{k:8s} {v['acc_flex']:8.1f} {v['acc_strict']:10.1f} {v['extract_rate']:9.1f} {v['truncation_rate']:7.1f} {v['mean_tokens']:7d} {v['minutes']:8.1f}")
print("SFT:", SFT); print("GRPO:", GRPO)
REPORT = dict(notebook = "11_sft_grpo_unsloth", mode = MODE, model = MODEL_NAME, load_in_4bit = LOAD_IN_4BIT,
              q = Q, dolci = DOLCI_DATASET, data = DATA, sft = SFT, grpo = GRPO, quick_eval = RESULTS, versions = VERSIONS)
os.makedirs("outputs", exist_ok = True)
json.dump(REPORT, open("outputs/report_11.json", "w"), indent = 1)
print("=" * 30, "把下面整段 JSON 贴回给 Claude", "=" * 30)
print(json.dumps(REPORT, ensure_ascii = False))

## 6.（可选）合并保存 → 推到 Hub → 用 `main` 分支的 `01` 做正式评测

`merged_16bit` 会把 LoRA 合并进权重并存成 fp16（排行榜要求）。推到 Hub 之后，
**换一个新的 runtime** 打开 [`01_eval_gpqa.ipynb`](https://colab.research.google.com/github/yy726/GPQA-eval/blob/main/notebooks/01_eval_gpqa.ipynb)，
`MODEL` 填你的仓库名、`BASE` 填上面的 `BASE_MODEL`。

In [ ]:
PUSH = False
HUB_REPO = "YOUR_HF_USERNAME/" + MODEL_NAME.split("/")[-1].lower() + "-dolci-sft-grpo"
if PUSH:
    model.push_to_hub_merged(HUB_REPO, tokenizer, save_method = "merged_16bit", token = os.environ.get("HF_TOKEN"))
    print(f"pushed: https://huggingface.co/{HUB_REPO}")
else:
    model.save_pretrained_merged("outputs/merged_16bit", tokenizer, save_method = "merged_16bit")
    print("saved locally: outputs/merged_16bit")